# Removing the vision encoder from a multimodal model

This is the companion to the notebook where we kept the encoder. Today we do the opposite:
throw the encoder away and keep the language model.

Same starting point, `zai-org/GLM-4.6V-Flash`, which is a GLM-4 language model with an
image encoder bolted on. Last time we walked away with 1.8 GB of encoder. This time we walk
away with 18.8 GB of language model that you can actually talk to.

Why would you want this? Three reasons, in increasing order of honesty:

1. It saves the 1.8 GB the encoder occupies.
2. You get a checkpoint that loads as a plain causal LM, with no multimodal machinery.
3. Mostly, though: fine-tuning a multimodal checkpoint on text-only data hits awkward
   errors, because the training code expects image bookkeeping that isn't there. Strip the
   vision path and those problems disappear.

The interesting part of this notebook is not the surgery. The surgery is a dictionary
filter, same as last time. The interesting part is **proving we didn't break anything**,
because here there is a genuine risk that we might.

A note on hardware: this one does not run on a laptop. We hold a 20 GB model in GPU memory
to get a reference, so you want something with 40 GB. Last time we got away with a laptop
because the encoder was small and lived in a single file.

Also worth saying up front: the outputs saved in this notebook come from a real run on an
A100. If you re-run it yourself the numbers should match exactly, and if they don't, that's
worth investigating rather than shrugging at.

## Setting up

The assert on compute capability is doing real work here, so it's worth a word.

These weights are stored in `bfloat16`. GPUs from NVIDIA's Ampere generation onward handle
that format natively. Older cards don't, and would silently convert to something else. Our
entire verification strategy rests on comparing numbers for *exact* equality, so a silent
format conversion would wreck it. Better to refuse to run.

In [1]:
import os, json, shutil, gc, time
from pathlib import Path
from collections import defaultdict

import torch, transformers
from safetensors import safe_open
from safetensors.torch import load_file, save_file

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    # sm_XY is the compute capability. 80 and above means Ampere or newer.
    print(f"gpu          {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")
    # Our checks compare floats for exact equality. A card without native bfloat16
    # would convert behind our backs and the comparison would stop meaning anything.
    assert p.major >= 8, "bf16 needs Ampere (sm_80+). Pre-Ampere breaks the exactness check."

REPO = "zai-org/GLM-4.6V-Flash"
SRC  = Path.home() / "models" / "GLM-4.6V-Flash"      # the original, all four shards
DST  = Path.home() / "models" / "GLM-4.6-Flash-text"  # what we are about to build
DST.parent.mkdir(parents=True, exist_ok=True)

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch        2.7.0
transformers 5.16.0.dev0
cuda         True
gpu          NVIDIA A100-SXM4-40GB  42.4 GB  sm_80


## Step 1. Download the whole thing

In the encoder notebook we were clever and downloaded one shard out of four, because every
tensor we wanted happened to live in that one file.

No such luck here. We're keeping 91% of the model, so we need all of it. 20.6 GB.

Take a moment on the config that prints out. `architectures` says
`Glm4vForConditionalGeneration` — "conditional generation" is the naming convention for
"generates text, conditioned on something else", the something else being an image. Our job
is to turn that into plain `Glm4ForCausalLM`, a language model conditioned on nothing but
its own text.

In [2]:
from huggingface_hub import snapshot_download

# snapshot_download grabs the whole repo. 20.6 GB, so give it a few minutes.
# max_workers parallelises the shards.
t0 = time.time()
snapshot_download(repo_id=REPO, local_dir=SRC, max_workers=8)
print(f"downloaded in {time.time()-t0:.0f}s")

cfg = json.loads((SRC / "config.json").read_text())
print("\narchitectures :", cfg["architectures"])
print("model_type    :", cfg["model_type"])
print("tie_word_embeddings:", cfg.get("tie_word_embeddings"))
print("\ntext_config:")
for k in ("hidden_size","intermediate_size","num_hidden_layers","num_attention_heads",
          "num_key_value_heads","vocab_size","max_position_embeddings","rms_norm_eps",
          "attention_bias","hidden_act","rope_parameters"):
    print(f"  {k:<24} {cfg['text_config'].get(k)}")
print("\nvision_config:")
for k in ("depth","hidden_size","num_heads","patch_size","image_size",
          "spatial_merge_size","temporal_patch_size","out_hidden_size"):
    print(f"  {k:<24} {cfg['vision_config'].get(k)}")


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]


Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 711.39it/s]

downloaded in 0s

architectures : ['Glm4vForConditionalGeneration']
model_type    : glm4v
tie_word_embeddings: False

text_config:
  hidden_size              4096
  intermediate_size        13696
  num_hidden_layers        40
  num_attention_heads      32
  num_key_value_heads      2
  vocab_size               151552
  max_position_embeddings  131072
  rms_norm_eps             1e-05
  attention_bias           True
  hidden_act               silu
  rope_parameters          {'mrope_section': [8, 12, 12], 'partial_rotary_factor': 0.5, 'rope_theta': 500000, 'rope_type': 'default'}

vision_config:
  depth                    24
  hidden_size              1536
  num_heads                12
  patch_size               14
  image_size               336
  spatial_merge_size       2
  temporal_patch_size      2
  out_hidden_size          4096


Two values in that output matter later, so flag them now.

`tie_word_embeddings: false`. Many models save memory by reusing the input embedding matrix
as the output layer — one matrix, two jobs, "tied". This one does not. It has a separate
`lm_head`, all 620 million parameters of it. If we forget to copy it, the model loses its
ability to produce words and we'd only find out at generation time.

`rope_parameters` contains an `mrope_section` entry. That's the multimodal position
encoding, and it's the one thing in this notebook that could genuinely go wrong. Step 5
deals with it.

## Step 2. Measure before you cut

Same idea as the encoder notebook: read the tensor names, sort them into vision and
not-vision, count parameters.

The asserts pin down what we expect to find. If a future version of the model has a
different number of vision tensors, this notebook should stop and make us look, not carry
on and produce something subtly wrong.

In [3]:
shards = sorted(SRC.glob("model-*.safetensors")) or sorted(SRC.glob("*.safetensors"))
shapes, dtypes = {}, {}
for s in shards:
    # Reading the header only -- no weights come into memory here.
    with safe_open(s, framework="pt") as f:
        for k in f.keys():
            sl = f.get_slice(k)
            shapes[k] = tuple(sl.get_shape())
            dtypes[k] = sl.get_dtype()

def numel(shape):
    n = 1
    for d in shape: n *= d
    return n

# The prefix is the whole basis of the split. Everything under model.visual.
# belongs to the encoder; everything else belongs to the language model.
VISION_PREFIX = "model.visual."
vision_keys = sorted(k for k in shapes if k.startswith(VISION_PREFIX))
text_keys   = sorted(k for k in shapes if not k.startswith(VISION_PREFIX))

n_vision = sum(numel(shapes[k]) for k in vision_keys)
n_text   = sum(numel(shapes[k]) for k in text_keys)
n_total  = n_vision + n_text

print(f"tensors       {len(shapes)}   (vision {len(vision_keys)} / text {len(text_keys)})")
print(f"TOTAL         {n_total:>15,}")
print(f"  vision      {n_vision:>15,}  {100*n_vision/n_total:6.3f}%   {n_vision*2/1e9:.3f} GB bf16")
print(f"  text        {n_text:>15,}  {100*n_text/n_total:6.3f}%   {n_text*2/1e9:.3f} GB bf16")

# If a re-upload changes the model, fail loudly here rather than producing
# a quietly wrong checkpoint further down.
assert n_vision == 892_498_432, f"expected 892,498,432 vision params, got {n_vision:,}"
assert len(vision_keys) == 181, f"expected 181 vision tensors, got {len(vision_keys)}"
print("\nmatches the published breakdown")

tensors       704   (vision 181 / text 523)
TOTAL          10,292,777,472
  vision          892,498,432   8.671%   1.785 GB bf16
  text          9,400,279,040  91.329%   18.801 GB bf16

matches the published breakdown


181 tensors, 8.7% of the model. That's what we're deleting. The remaining 523 tensors and
9.4 billion parameters are the language model.

Here's the same breakdown by submodule, if you want to see what a vision tower is made of.

In [4]:
# Vision submodules, so you can see what the 892M is actually made of.
import re
groups = defaultdict(lambda: [0, 0])
for k in vision_keys:
    tail = k[len(VISION_PREFIX):]
    g = re.sub(r"blocks\.\d+\..*", "blocks.*  (24 x ViT)", tail)
    g = re.sub(r"\.(weight|bias)$", "", g)
    groups[g][0] += numel(shapes[k]); groups[g][1] += 1

for g, (p, c) in sorted(groups.items(), key=lambda x: -x[1][0]):
    print(f"  {g:<38} {p:>13,}  ({c} tensors)")

  blocks.*  (24 x ViT)                     679,550,976  (168 tensors)
  merger.down_proj                          56,098,816  (1 tensors)
  merger.gate_proj                          56,098,816  (1 tensors)
  merger.up_proj                            56,098,816  (1 tensors)
  downsample                                25,169,920  (2 tensors)
  merger.proj                               16,777,216  (1 tensors)
  patch_embed.proj                           1,807,872  (2 tensors)
  embeddings.position_embedding                884,736  (1 tensors)
  merger.post_projection_norm                    8,192  (2 tensors)
  post_conv_layernorm                            1,536  (1 tensors)
  post_layernorm                                 1,536  (1 tensors)


## Step 3. Take a reference reading first

This is the step I want you to remember from this notebook.

We are about to modify a model. Afterwards we'll want to know whether we changed its
behaviour. That is only answerable if we recorded the behaviour *beforehand*.

So: before touching anything, run some text prompts through the original model and save the
raw output numbers. Logits, not generated text — logits are the model's full opinion about
every possible next token, so comparing them is far stricter than comparing whatever words
came out.

The prompts are deliberately varied: facts, code, reasoning, a number sequence, translation.
If our surgery damaged one capability and not others, a narrow prompt set could miss it.

Do this in your own work. Capture the "before" while you still can. Once you've overwritten
something, no amount of staring at the "after" tells you what changed.

In [5]:
PROMPTS = [
    "The capital of France is",
    "def fibonacci(n):",
    "Explain why the sky appears blue, in one sentence.",
    "1, 1, 2, 3, 5, 8, 13,",
    "Translate to German: The weather is cold today.",
    "The three laws of thermodynamics state that",
]

from transformers import AutoTokenizer, Glm4vForConditionalGeneration

# The tokenizer turns text into the integer IDs the model consumes.
tok = AutoTokenizer.from_pretrained(SRC, trust_remote_code=True)

t0 = time.time()
mm = Glm4vForConditionalGeneration.from_pretrained(
    SRC, dtype=torch.bfloat16, low_cpu_mem_usage=True
).to("cuda").eval()
print(f"loaded in {time.time()-t0:.0f}s")
print(f"params  {sum(p.numel() for p in mm.parameters()):,}")
print(f"vram    {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Logits, not generated text. Logits are the model's score for every token in the
# vocabulary, so comparing them catches differences that generated words would hide.
baseline = {}
with torch.no_grad():
    for p in PROMPTS:
        ids = tok(p, return_tensors="pt").to("cuda")
        # .float() promotes from bfloat16 so later subtraction is not itself lossy,
        # and .cpu() moves it off the GPU because we are about to need that memory.
        baseline[p] = mm(**ids).logits.float().cpu()
print(f"\ncaptured {len(baseline)} reference logit tensors")


Loading weights:   0%|          | 0/704 [00:00<?, ?it/s]


Loading weights:  78%|███████▊  | 550/704 [00:00<00:00, 5497.12it/s]


Loading weights: 100%|██████████| 704/704 [00:00<00:00, 5676.17it/s]

loaded in 4s
params  10,292,777,472
vram    20.64 GB



captured 6 reference logit tensors


Now free the GPU. Two models of this size will not sit in memory at once, so everything from
here on loads one at a time.

`del` alone is not enough. Python drops its reference, but PyTorch keeps the GPU memory in a
cache for reuse, so you need `empty_cache()` to actually hand it back. This trips up
everyone at least once.

In [6]:
del mm
gc.collect(); torch.cuda.empty_cache()
print(f"vram after free: {torch.cuda.memory_allocated()/1e9:.2f} GB")

vram after free: 0.01 GB


## Step 4. The surgery

Three operations on the dictionary of weights.

**Drop** anything whose name starts with `model.visual.` — the encoder.

**Rename** `model.language_model.something` to `model.something`. In the multimodal model the
language model is nested one level down, alongside the encoder. In a standalone causal LM
it's at the top. Chopping out that middle path segment is what makes the file loadable by
the plain class.

**Keep** `lm_head.weight`. Remember `tie_word_embeddings: false` from step 1. Nothing in the
code would warn you about dropping it, so it's on you to know.

The loop writes output in roughly 5 GB pieces and frees each input shard as it goes. That
keeps peak memory to about one shard rather than the whole model.

In [7]:
DST.mkdir(parents=True, exist_ok=True)

MAX_SHARD_BYTES = 5_000_000_000
weight_map, buf, buf_bytes, out_idx, total_bytes = {}, {}, 0, 1, 0
written, dropped = 0, 0
out_files = []

def flush():
    # Write the buffer out as one shard and start a fresh one.
    global buf, buf_bytes, out_idx
    if not buf: return
    name = f"model-{out_idx:05d}.safetensors"
    save_file(buf, DST / name, metadata={"format": "pt"})
    for k in buf: weight_map[k] = name
    out_files.append(name)
    print(f"  wrote {name}  ({len(buf)} tensors, {buf_bytes/1e9:.2f} GB)")
    buf, buf_bytes = {}, 0
    out_idx += 1

for s in shards:
    sd = load_file(s)
    for k in sorted(sd):
        if k.startswith(VISION_PREFIX):
            dropped += 1              # the encoder: discard
            continue
        # Un-nest the language model: model.language_model.layers.0 -> model.layers.0
        nk = k.replace("model.language_model.", "model.", 1) if k.startswith("model.language_model.") else k
        # .contiguous() guarantees a clean memory layout; safetensors requires it.
        t = sd[k].contiguous()
        nbytes = t.numel() * t.element_size()
        if buf_bytes + nbytes > MAX_SHARD_BYTES:
            flush()
        buf[nk] = t; buf_bytes += nbytes; total_bytes += nbytes; written += 1
    del sd; gc.collect()              # release this shard before reading the next
flush()

print(f"\nkept {written} tensors, dropped {dropped}")
assert dropped == 181, f"expected to drop 181 vision tensors, dropped {dropped}"

# safetensors needs an index saying which tensor lives in which output shard.
(DST / "model.safetensors.index.json").write_text(json.dumps(
    {"metadata": {"total_size": total_bytes}, "weight_map": weight_map}, indent=2))
print(f"index written · total_size {total_bytes:,} bytes ({total_bytes/1e9:.2f} GB)")

  wrote model-00001.safetensors  (119 tensors, 4.91 GB)


  wrote model-00002.safetensors  (156 tensors, 4.90 GB)


  wrote model-00003.safetensors  (156 tensors, 4.90 GB)


  wrote model-00004.safetensors  (92 tensors, 4.10 GB)



kept 523 tensors, dropped 181
index written · total_size 18,800,558,080 bytes (18.80 GB)


A quick structural check before going further. Three things should be true: no vision
tensors survived, no name still has the old nesting, and the output head is present.

Cheap to check, and each one is a bug that would otherwise surface much later and much more
confusingly.

In [8]:
# Confirm the resulting key layout is exactly what Glm4ForCausalLM expects.
# Keep only the first two path segments of each name, so we see the shape of the
# model rather than 523 individual tensors.
prefixes = sorted({k.split(".")[0] + "." + (k.split(".")[1] if len(k.split(".")) > 1 else "")
                   for k in weight_map})
print("top-level prefixes:", prefixes)
assert not any(k.startswith("model.visual") for k in weight_map)
assert not any(k.startswith("model.language_model") for k in weight_map)
assert "lm_head.weight" in weight_map, "lm_head missing -- it is untied and must be kept"
print("layout OK")

top-level prefixes: ['lm_head.weight', 'model.embed_tokens', 'model.layers', 'model.norm']
layout OK


## Step 5. The config, and the one real risk

Everything so far has been bookkeeping. This is the part with an actual judgement call in
it.

The original config has a field called `mrope_section: [8, 12, 12]`. That's *multimodal*
rotary position embedding. Ordinary RoPE tells the model where each token sits in the
sequence. Multimodal RoPE splits that job three ways — position in time, position vertically
in the image, position horizontally — so the model can tell that one patch is above another.

We're removing images, so those axes have nothing to describe. The plan is to drop the field
and let the model use ordinary RoPE.

**Is that safe?** Here's the argument. When the input is pure text there are no image
patches, so all three coordinates are set to the same value: the token's position in the
sequence. Three identical numbers going into a mechanism that combines three numbers gives
the same result as one number going into the ordinary mechanism. The two should be
identical.

That is an argument from reading the code. Arguments from reading the code are wrong all the
time. Step 6 is where we find out.

Notice we build the config through `Glm4Config` rather than hand-editing JSON. And once
again `architectures` has to be set manually — the config class won't do it, and without it
`AutoModelForCausalLM` has no idea which class to construct.

In [9]:
from transformers import Glm4Config

tc = cfg["text_config"]
rope = dict(tc.get("rope_parameters") or {})
# Drop the multimodal position axes. With no image tokens they have nothing to encode.
mrope = rope.pop("mrope_section", None)
print("dropped mrope_section:", mrope)

kwargs = dict(
    vocab_size=tc["vocab_size"],
    hidden_size=tc["hidden_size"],
    intermediate_size=tc["intermediate_size"],
    num_hidden_layers=tc["num_hidden_layers"],
    num_attention_heads=tc["num_attention_heads"],
    num_key_value_heads=tc["num_key_value_heads"],
    hidden_act=tc.get("hidden_act", "silu"),
    max_position_embeddings=tc["max_position_embeddings"],
    rms_norm_eps=tc["rms_norm_eps"],
    attention_bias=tc.get("attention_bias", True),
    tie_word_embeddings=cfg.get("tie_word_embeddings", False),
    pad_token_id=tc.get("pad_token_id"),
    eos_token_id=tc.get("eos_token_id"),
    dtype="bfloat16",
)

# Building through the config class rather than editing JSON means renamed fields
# in a future transformers release surface as an error here, not as silent nonsense.
new_cfg = Glm4Config(**kwargs)

# Carry RoPE across under whichever field this transformers version uses.
if hasattr(new_cfg, "rope_parameters"):
    new_cfg.rope_parameters = rope
else:
    new_cfg.rope_theta = rope.get("rope_theta", 500000)
    new_cfg.partial_rotary_factor = rope.get("partial_rotary_factor", 0.5)

# Glm4Config does not populate this, and without it AutoModelForCausalLM has no
# class to dispatch to -- the checkpoint would only load via the explicit class.
new_cfg.architectures = ["Glm4ForCausalLM"]

new_cfg.save_pretrained(DST)
print(json.dumps(json.loads((DST / "config.json").read_text()), indent=2)[:1200])

dropped mrope_section: [8, 12, 12]
{
  "architectures": [
    "Glm4ForCausalLM"
  ],
  "attention_bias": true,
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "dtype": "bfloat16",
  "eos_token_id": [
    151329,
    151336,
    151338
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 13696,
  "max_position_embeddings": 131072,
  "model_type": "glm4",
  "num_attention_heads": 32,
  "num_hidden_layers": 40,
  "num_key_value_heads": 2,
  "pad_token_id": 151329,
  "partial_rotary_factor": 0.5,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "partial_rotary_factor": 0.5,
    "rope_theta": 500000,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.16.0.dev0",
  "use_cache": true,
  "vocab_size": 151552
}


The tokenizer comes across unchanged.

Its vocabulary still contains the special tokens that used to mark where an image went. We
leave them. They cost nothing, removing them would shift every token ID after them, and the
model simply never emits them now. Deleting things because they look untidy is a good way to
break a tokenizer.

The image preprocessing configs do get deleted, because a text-only model that ships
instructions for resizing JPEGs is just confusing.

In [10]:
# Tokenizer + chat template travel unchanged; image/video ids stay in the 151552 vocab
# (harmless -- they are simply never emitted). Preprocessing configs are dropped.
for f in ("tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
          "vocab.txt", "merges.txt", "chat_template.jinja", "generation_config.json"):
    src = SRC / f
    if src.exists():
        shutil.copy2(src, DST / f); print("copied", f)

for f in ("preprocessor_config.json", "processor_config.json", "video_preprocessor_config.json"):
    (DST / f).unlink(missing_ok=True)

print("\nfinal contents:")
for p in sorted(DST.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size/1e6:>10.1f} MB")

copied tokenizer.json
copied tokenizer_config.json
copied chat_template.jinja
copied generation_config.json

final contents:
  chat_template.jinja                             0.0 MB
  config.json                                     0.0 MB
  generation_config.json                          0.0 MB
  model-00001.safetensors                      4913.0 MB
  model-00002.safetensors                      4895.3 MB
  model-00003.safetensors                      4895.3 MB
  model-00004.safetensors                      4097.1 MB
  model.safetensors.index.json                    0.0 MB
  tokenizer.json                                 20.0 MB
  tokenizer_config.json                           0.0 MB


## Step 6. The moment of truth

Load our new checkpoint, run the same six prompts, compare against the baseline from step 3.

What counts as passing? Not "close". **Exactly zero difference.**

That sounds harsh but it's the right bar, and here's why. We did no arithmetic on the
weights. We copied them and renamed their keys. If the model computes the same function, the
floating-point operations are identical in the same order and must produce identical bits.
Any difference at all — even one in the twelfth decimal place — means the computation graph
changed somewhere, and we'd want to know where before trusting it.

A tiny non-zero difference is the worst outcome, worse than a big one. Big differences you
notice. Tiny ones look like harmless rounding, ship, and then behave oddly six months later
in some situation you never tested.

In [11]:
from transformers import Glm4ForCausalLM

t0 = time.time()
txt = Glm4ForCausalLM.from_pretrained(DST, dtype=torch.bfloat16, low_cpu_mem_usage=True).to("cuda").eval()
print(f"loaded in {time.time()-t0:.0f}s")
print(f"params  {sum(p.numel() for p in txt.parameters()):,}")
print(f"vram    {torch.cuda.memory_allocated()/1e9:.2f} GB\n")

worst = 0.0
with torch.no_grad():
    for p in PROMPTS:
        ids = tok(p, return_tensors="pt").to("cuda")
        new = txt(**ids).logits.float().cpu()
        # Largest disagreement anywhere in the logit tensor.
        # Biggest absolute disagreement across the whole logit tensor -- the
        # strictest single number we can put on "did the model change".
        d = (baseline[p] - new).abs().max().item()
        # And a coarser check: would both models pick the same next token?
        same = baseline[p].argmax(-1).eq(new.argmax(-1)).all().item()
        worst = max(worst, d)
        print(f"  max|d|={d:<12.3e}  argmax_match={same}   {p[:42]!r}")

print()
if worst == 0.0:
    print("EXACT MATCH -- mRoPE collapsed to RoPE cleanly. Extraction is lossless.")
else:
    print(f"DIVERGENCE {worst:.3e} -- do NOT ship this checkpoint.")
    print("Inspect position_ids handling in Glm4vTextModel vs Glm4Model before continuing.")


Loading weights:   0%|          | 0/523 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 523/523 [00:00<00:00, 6236.48it/s]

loaded in 4s
params  9,400,279,040
vram    18.85 GB

  max|d|=0.000e+00     argmax_match=True   'The capital of France is'
  max|d|=0.000e+00     argmax_match=True   'def fibonacci(n):'
  max|d|=0.000e+00     argmax_match=True   'Explain why the sky appears blue, in one s'


  max|d|=0.000e+00     argmax_match=True   '1, 1, 2, 3, 5, 8, 13,'
  max|d|=0.000e+00     argmax_match=True   'Translate to German: The weather is cold t'
  max|d|=0.000e+00     argmax_match=True   'The three laws of thermodynamics state tha'

EXACT MATCH -- mRoPE collapsed to RoPE cleanly. Extraction is lossless.


Zero across the board. The argument from step 5 held: with text-only input, multimodal RoPE
really does reduce to ordinary RoPE.

Now stop and think about whether we've actually finished, because we haven't.

Every one of those prompts is under a dozen tokens. We just removed part of the *position
encoding* system. A bug in position handling might not show up until positions get large —
the first few can be fine while things drift apart later in a long sequence.

Testing something with inputs too small to stress the thing you changed is a classic way to
get a green light you haven't earned.

## Step 7. Try it again, but longer

Roughly 1200 tokens this time.

Note the loading dance: we run our model, free it, then load the original to get the
reference. Two 20 GB models won't fit side by side, so the memory management is part of the
test.

In [12]:
# Repetition is fine here. We are testing position handling, not comprehension,
# so all that matters is that the sequence gets long.
long_prompt = ("The following is a technical description repeated for length. " * 120) + \
              "In summary, the key point is"
ids = tok(long_prompt, return_tensors="pt").to("cuda")
print("sequence length:", ids["input_ids"].shape[1])

with torch.no_grad():
    new_long = txt(**ids).logits.float().cpu()

# Free ours before loading the original -- both will not fit at once.
del txt; gc.collect(); torch.cuda.empty_cache()

mm = Glm4vForConditionalGeneration.from_pretrained(
    SRC, dtype=torch.bfloat16, low_cpu_mem_usage=True).to("cuda").eval()
with torch.no_grad():
    ref_long = mm(**ids).logits.float().cpu()
del mm; gc.collect(); torch.cuda.empty_cache()

d = (ref_long - new_long).abs().max().item()
print(f"long-context max|d| = {d:.3e}")
print("PASS" if d == 0.0 else "FAIL -- position encoding diverges at length")

sequence length: 1207



Loading weights:   0%|          | 0/704 [00:00<?, ?it/s]


Loading weights:  66%|██████▌   | 462/704 [00:00<00:00, 4617.62it/s]


Loading weights: 100%|██████████| 704/704 [00:00<00:00, 5503.98it/s]

long-context max|d| = 0.000e+00
PASS


Still exactly zero, at 1207 tokens. Now the result means something.

Two checks, and the second is the one that earns the claim. The first alone would have been
a weak test dressed up as a strong one.

## Step 8. Actually talk to it

Numbers matching is necessary but it's nice to see words come out.

One transformers detail worth knowing, since it bites people: `apply_chat_template` returns
a dictionary-like object, not a plain tensor. Pass it to `generate()` positionally and you
get a confusing `AttributeError` about `.shape`. Unpack it with `**` instead, which also
carries the attention mask through.

In [13]:
txt = Glm4ForCausalLM.from_pretrained(DST, dtype=torch.bfloat16, low_cpu_mem_usage=True).to("cuda").eval()

msgs = [{"role": "user", "content": "In two sentences, what is a mixture-of-experts layer?"}]
# return_dict=True: in transformers v5 apply_chat_template yields a BatchEncoding,
# and passing it positionally to generate() makes it look for .shape on a dict.
# Unpacking also carries the attention_mask through, which the tensor form drops.
enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                              return_tensors="pt", return_dict=True).to("cuda")

# do_sample=False means greedy decoding: always take the highest-scoring token.
# Deterministic, so two runs of this cell give the same text.
with torch.no_grad():
    out = txt.generate(**enc, max_new_tokens=160, do_sample=False)
print(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))


Loading weights:   0%|          | 0/523 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 523/523 [00:00<00:00, 6295.48it/s]

<think>用户现在需要解释混合专家层（mixture-of-experts layer）是什么，用两个句子。首先得准确定义，然后说明其作用或结构。

首先，混合专家层是一种神经网络结构，它将输入分配给多个独立的“专家”子网络（专家），每个专家处理特定类型的输入，然后通过一个门控机制（如softmax加权）组合这些专家的输出。然后，说明其目的，比如提升模型在处理复杂任务时的效率和泛化能力，通过让不同专家专注不同模式来提高性能。

现在组织语言，确保两个句子，清晰简洁。

第一句定义结构：A mixture-of-experts (MoE) layer is a neural network component that routes an input to multiple specialized “expert” sub-networks, each designed to


It answers, though it gets cut off mid-thought.

That's not a bug in our surgery. This model reasons out loud before answering — it writes a
scratchpad first, sometimes in Chinese regardless of the language you asked in, then gives
its actual reply. 160 tokens isn't enough to get through the thinking. Ask for 600 and
you'll see a real answer.

Worth flagging because it looks like breakage the first time you see it, and isn't.

## Step 9. What we ended up with

In [14]:
src_bytes = sum(p.stat().st_size for p in SRC.glob("*.safetensors"))
dst_bytes = sum(p.stat().st_size for p in DST.glob("*.safetensors"))

print(f"{'':<14}{'params':>18}{'weights':>14}")
print(f"{'original':<14}{n_total:>18,}{src_bytes/1e9:>12.2f} GB")
print(f"{'text-only':<14}{n_text:>18,}{dst_bytes/1e9:>12.2f} GB")
print(f"{'removed':<14}{n_vision:>18,}{(src_bytes-dst_bytes)/1e9:>12.2f} GB"
      f"   ({100*n_vision/n_total:.3f}%)")
print(f"\ncheckpoint: {DST}")
print("\nNext: quantize (the far bigger lever) or push to the Hub.")

                          params       weights
original          10,292,777,472       20.59 GB
text-only          9,400,279,040       18.80 GB
removed              892,498,432        1.79 GB   (8.671%)

checkpoint: /home/ubuntu/models/GLM-4.6-Flash-text

Next: quantize (the far bigger lever) or push to the Hub.


## Wrapping up

A working 9.4 billion parameter language model, 1.8 GB smaller than what we started with,
provably identical on text.

Compare the two notebooks. When we kept the encoder we got a component: something that turns
images into vectors and needs a partner to be useful. Here we got a finished model you can
load and talk to. Same model, cut in two, and only one half stands on its own — because the
half with the output head is the half that can produce something a human reads.

If you take one habit away, make it the shape of steps 3, 6 and 7: record the behaviour
before you change anything, compare afterwards, and then ask whether your test was actually
hard enough. Getting zero on the short prompts felt like success. It wasn't yet.

### Things to try

1. Rerun step 8 with `max_new_tokens=600` and read the full answer.
2. Comment out the `lm_head.weight` assert in step 4 and drop that tensor. What breaks, and
   at what point do you find out?
3. Keep `mrope_section` in the config instead of dropping it. Does step 6 still pass? Should
   it?
4. The saving here is 1.8 GB out of 20.6. Look up what 4-bit quantization does to the same
   model, and decide which one you'd actually reach for if the goal were a smaller file.